# 1,000 long student essays from Gemini 3.5 Flash-Lite (Batch API)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohaaa825/Skolfie/blob/main/notebooks/generate_gemini_batch.ipynb)

Asks `gemini-3.5-flash-lite` for `N_ESSAYS` student assignments of 800-1,500 words through the
**Gemini Batch API** (half the normal token price; results usually within hours, at most 24 h).

- **Variety**: a seeded plan crosses 14 genres (argumentative essay, debate article, persuasive
  letter, speech, compare/contrast, cause and effect, explanatory essay, research paper, lab
  report, book report, literary analysis, critique, personal narrative, reflection) with topics
  that fit each genre, a grade level (grade 8 to upper-level university) and a student voice
  (14 personas, some asking "make it sound like a real student"). No genre+topic pair repeats.
  The 15 PERSUADE assignments come first in the genres that fit them, since those are the topics
  of the human student essays.
- **Length**: every prompt asks for a word count between 850 and 1,450 and says it must stay
  within 800-1,500. Each essay is cleaned (Markdown marks removed) and counted; out-of-range or
  cut-off essays are asked once more with a note on the length, then dropped, and the next
  unused assignments top up until exactly `N_ESSAYS` are kept.
- **Thinking**: level `minimal` (the model's default, set explicitly). No temperature (Gemini 3.x
  ignores it).
- **Context caching**: tried for the shared instruction. That instruction is only about 450
  tokens, which is probably below the model's minimum cache size; if the API refuses, the
  notebook says so and sends the instruction with each request (it is under 5% of the cost).
- **Cost**: about 2.5 USD for 1,000 essays at batch prices; step 3 prints the estimate and the run
  stops before any round that would pass `MAX_USD`.

**Before you start**
1. A Gemini API key from Google AI Studio (aistudio.google.com -> Get API key) on a project with
   billing turned on (the Batch API is a paid-tier feature).
2. In Colab: the key icon on the left (Secrets) -> add `GEMINI_API_KEY`, allow notebook access.
   The key never goes into a file or the repo.
3. A CPU runtime is enough (Runtime -> Change runtime type -> CPU).
4. **Runtime -> Run all**. If Colab disconnects while the job is waiting, run all again: the job
   name is saved in Drive (`WORK_DIR/state.json`), so it waits for the same job instead of
   paying twice.

Privacy: only the assignments go to Google, never a student's text.

## 1. Settings

In [ ]:
DRIVE = "/content/drive/MyDrive/skolfie"
OUT_DIR = f"{DRIVE}/data/raw/fresh_v3"               # retrain_modern.ipynb reads essays from here
WORK_DIR = f"{DRIVE}/data/raw/gemini_batch_work"     # job state, requests, raw results, rejects

N_ESSAYS = 1000
SEED = 0                    # the same seed gives the same assignments
MODEL = "gemini-3.5-flash-lite"
THINKING_LEVEL = "minimal"  # "minimal" | "low" | "medium" | "high"
USE_CACHE = True            # try context caching for the shared instruction
MAX_USD = 10.0              # hard cap on the whole run (all rounds)
MAX_ROUNDS = 4              # round 1 + up to 3 top-up rounds
MAX_WAIT_HOURS = 30         # stop waiting after this; run again later to keep waiting

REPO = "mohaaa825/Skolfie"
BRANCH = "main"

## 2. Mount Drive, clone, install, read the key

In [ ]:
import os
import subprocess
import sys

from google.colab import drive

drive.mount("/content/drive")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)

def secret(name):
    try:
        from google.colab import userdata

        return userdata.get(name)
    except Exception:
        return None


def git(*args, check=True):
    """Run git; a GITHUB_TOKEN secret reaches it through a credential helper that reads it from
    the environment, so the token never appears in a URL, a command line, an error message or
    .git/config."""
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    auth = []
    token = secret("GITHUB_TOKEN")
    if token:
        env["SKOLFIE_GITHUB_TOKEN"] = token
        auth = ["-c", "credential.helper=", "-c",
                "credential.helper=!f() { test \"$1\" = get && echo username=x-access-token "
                "&& echo \"password=$SKOLFIE_GITHUB_TOKEN\"; }; f"]
    return subprocess.run(["git", *auth, *args], env=env, check=check)


url = f"https://github.com/{REPO}.git"
if os.path.isdir("/content/Skolfie/.git"):
    # Clones made by older versions of this notebook kept the token in the remote URL.
    git("-C", "/content/Skolfie", "remote", "set-url", "origin", url)
    git("-C", "/content/Skolfie", "pull", "--ff-only")
else:
    git("clone", "--depth", "1", "--branch", BRANCH, url, "/content/Skolfie")

%cd /content/Skolfie
!git log --oneline -1
!pip install -q -e . "google-genai>=1.30"

key = secret("GEMINI_API_KEY")
assert key, ("Add the secret GEMINI_API_KEY (key icon on the left) and allow this "
             "notebook to read it.")
os.environ["GEMINI_API_KEY"] = key   # only in this runtime's memory
del key
print("Key found.")

## 3. The plan and the cost estimate

Nothing is sent in this step. Prices: 0.30 / 2.50 USD per 1M input / output tokens, halved for
batch (`skolfie/data/generation/gemini_batch.py`, `PRICES`); check them against
ai.google.dev/gemini-api/docs/pricing.

In [ ]:
def run(cmd):
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            bufsize=1)
    for line in proc.stdout:
        sys.stdout.write(line)
    return proc.wait()

ARGS = ["--out-dir", OUT_DIR, "--work-dir", WORK_DIR, "--n", str(N_ESSAYS), "--seed", str(SEED),
        "--model", MODEL, "--thinking-level", THINKING_LEVEL, "--max-usd", str(MAX_USD),
        "--max-rounds", str(MAX_ROUNDS), "--max-wait-hours", str(MAX_WAIT_HOURS)]
if not USE_CACHE:
    ARGS.append("--no-cache")
run([sys.executable, "-u", "scripts/gemini_batch_essays.py", *ARGS, "--dry-run"])

## 4. Run the batch jobs

Submits round 1 (`N_ESSAYS` + 10% spare), waits for it (checks once a minute), keeps the essays in
range, then sends retries and top-ups until `N_ESSAYS` are kept. Exit code 3 means the job is
still running after `MAX_WAIT_HOURS`: run this cell again later.

In [ ]:
code = run([sys.executable, "-u", "scripts/gemini_batch_essays.py", *ARGS])
print("exit code", code)

## 5. Check what came back

In [ ]:
import json

import pandas as pd

path = f"{OUT_DIR}/gemini-3.5-flash-lite-batch.jsonl"
recs = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]
words = pd.Series([len(r["text"].split()) for r in recs])
print(f"{len(recs)} essays; words min {words.min()}, median {int(words.median())}, "
      f"max {words.max()}")
print("outside 800-1500:", int(((words < 800) | (words > 1500)).sum()))
print("look like Markdown:", sum(("**" in r["text"]) or r["text"].lstrip().startswith("#")
                                 for r in recs))
state = json.load(open(f"{WORK_DIR}/state.json"))
print(f"spent ${state['spent_usd']:.2f} in {len(state['rounds'])} round(s)")
for r in state["rounds"]:
    print("  round", r["n"], r.get("counts"), r.get("failed_state", ""))
print("\nExample:\n")
print(recs[0]["text"][:1500])

## Next

Tell Claude in the project thread that this has finished; it checks the essays from Drive. The file
is named `gemini-3.5-flash-lite-batch.jsonl` (generator `gemini-3.5-flash-lite`, family `gemini`)
so it does not clash with the repo's earlier Gemini file when `retrain_modern.ipynb` adds it.